# GPU-Side Checkpointing for cuDF DataFrames

This notebook demonstrates FlowBook's GPU checkpoint mode, which keeps cudf
DataFrames on the GPU during checkpointing instead of converting to pandas (CPU).

**Key benefits:**
- ~400x faster checkpointing (~3ms vs ~1.3s for large DataFrames)
- Preserves compact dtypes (avoids int8 → float64 inflation)
- GPU-native diffing for change detection

**Trade-off:** Uses GPU memory for checkpoint storage instead of CPU memory.

## Setup: Create a Large cuDF DataFrame

We build a DataFrame mimicking a real Kaggle competition notebook:
- 2M rows, ~150 columns
- Mix of int8, float32, and float64 columns
- int8 columns are key: column-by-column CPU conversion inflates these to float64

In [ ]:
import cudf
import numpy as np
import time

# Build a large DataFrame with mixed dtypes
np.random.seed(42)
n_rows = 2_000_000
data = {}

# 15 factorized categoricals (float32)
for i in range(15):
    data[f'cat_{i}'] = np.random.randint(0, 50, size=n_rows).astype(np.float32)

# 10 numeric features (float64)
for i in range(10):
    data[f'feat_{i}'] = np.random.normal(0, 1, size=n_rows).astype(np.float64)

# 105 pair interaction features (int8) — C(15,2)
cats = [f'cat_{i}' for i in range(15)]
for i in range(len(cats)):
    for j in range(i + 1, len(cats)):
        data[f'{cats[i]}_{cats[j]}'] = np.random.randint(0, 127, size=n_rows).astype(np.int8)

# 10 digit features (int8)
for i in range(10):
    data[f'digit_{i}'] = np.random.randint(-1, 10, size=n_rows).astype(np.int8)

df = cudf.DataFrame(data)
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {len(df.columns)} ({sum(1 for c in df.columns if df[c].dtype == np.int8)} int8, "
      f"{sum(1 for c in df.columns if df[c].dtype == np.float32)} float32, "
      f"{sum(1 for c in df.columns if df[c].dtype == np.float64)} float64)")
print(f"GPU memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## Enable GPU Checkpoint Mode

By default, FlowBook converts cudf objects to pandas for checkpointing.
With `%cudf_gpu_checkpoint on`, checkpoints stay on the GPU.

In [ ]:
%cudf_gpu_checkpoint on

## Feature Engineering

Now we perform some feature engineering. Each cell execution triggers
a checkpoint. With GPU checkpoint mode on, the checkpoint stays on the GPU.

In [ ]:
# Add rolling statistics
df['feat_0_squared'] = df['feat_0'] ** 2
df['feat_1_abs'] = df['feat_1'].abs()
df['feat_sum'] = df['feat_0'] + df['feat_1'] + df['feat_2']
print(f"After feature engineering: {df.shape[1]} columns")

In [ ]:
# Create more derived features
df['cat_0_x_feat_0'] = df['cat_0'] * df['feat_0']
df['cat_1_x_feat_1'] = df['cat_1'] * df['feat_1']
df['digit_sum'] = sum(df[f'digit_{i}'] for i in range(10))
print(f"After more features: {df.shape[1]} columns")

In [ ]:
# Create a target variable and train/test split
target = (df['feat_0'] > 0).astype(np.int8)
train_mask = np.random.random(n_rows) < 0.8
train_df = df[train_mask]
test_df = df[~train_mask]
print(f"Train: {len(train_df):,} rows, Test: {len(test_df):,} rows")

## Modify the DataFrame to Trigger Change Detection

FlowBook tracks changes between checkpoints. In GPU checkpoint mode,
change detection uses cudf's native `.equals()` method on GPU.

In [ ]:
# Modify some columns — this will be detected by GPU-native diff
df['feat_0'] = df['feat_0'] * 2.0
df['cat_0'] = df['cat_0'] + 1
print("Modified feat_0 and cat_0")

In [ ]:
# Add new columns
df['new_feat_1'] = df['feat_3'] * df['feat_4']
df['new_feat_2'] = df['feat_5'].clip(-2, 2)
print(f"Final shape: {df.shape}")
print(f"GPU memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## Check Checkpoint Status

The FlowBook status output shows timing for each cell's checkpoint.
With GPU mode, `State` time (checkpoint creation) should be very fast.

In [ ]:
%flowbook_status

## Disable GPU Checkpoint Mode

You can switch back to CPU-side checkpointing at any time.

In [ ]:
%cudf_gpu_checkpoint off